# 03c · LLM labels for r/wallstreetbets comments (Qwen2.5-7B, no manual labeling)

**Why.** The 03b BiLSTM learned from news and tweets only, and it transfers poorly to Reddit. Here the same LLM teacher as 03a labels a sample of Reddit comments, so 03d can train on meme-stock language.

**Sample.** 2,000 comments per ticker (GME, AMC, BB), spread evenly over the collected days (Jan – Jul 2021). Comments shorter than 4 words, and deleted/removed ones, are dropped. Comments are cut to 600 characters.
* The BB file of 30 Nov – 8 Dec 2020 is **not sampled**: its 2,555 comments (scored by ChatGPT and Claude in `Comments_w_Labels`) stay an untouched external check, and Qwen labels them too.
* **Split by calendar day** (70 / 15 / 15 % of days → train / val / test), so comments from the same day (same news, same threads) never sit on both sides.

**Prompt `reddit_v1`.** Same classifier as 03a: next-token probabilities of bearish / neutral / bullish, ticker-specific, with a short note on WSB slang. The 03a calibration (T = 4, neutral weight w = 4) is applied, since there are no human Reddit labels to re-tune it.

**Runtime.** GPU (T4). About 4 comments/s → ~40 min. Results are checkpointed to Drive every 500 comments, so a disconnect loses at most one chunk.

**Output.** `DataSource/llm_labels_reddit.csv`

In [ ]:
# 0. Setup (Runtime > Change runtime type > T4 GPU)
!pip -q install -U bitsandbytes accelerate
from google.colab import drive
drive.mount('/content/drive')
import os, glob, re, time, numpy as np, pandas as pd, torch
from scipy import stats
assert torch.cuda.is_available(), "No GPU: Runtime > Change runtime type > T4 GPU"

def find_project_dir(name="dl-quant_project"):
    cands = [f"/content/drive/MyDrive/Colab Notebooks/{name}", f"/content/drive/MyDrive/{name}",
             *glob.glob(f"/content/drive/MyDrive/*/{name}"), *glob.glob(f"/content/drive/Shareddrives/*/{name}")]
    for p in cands:
        if os.path.isdir(p): return p
    raise FileNotFoundError("Add a shortcut to the shared 'dl-quant_project' folder in My Drive (see 00_START_HERE).")
DATA_DIR = os.path.join(find_project_dir(), "DataSource")
SEED = 42
N_PER_TICKER, MIN_WORDS, MAX_CHARS = 2000, 4, 600
pd.set_option("display.width", 200)
print("GPU:", torch.cuda.get_device_name(0), "| data:", DATA_DIR)

Mounted at /content/drive
GPU: Tesla T4 | data: /content/drive/MyDrive/Colab Notebooks/dl-quant_project/DataSource


## 1. Sample comments and split by day

In [ ]:
CHECK_FILE = "wallstreetbets_BB_Nov30_2020_Dec9_2020.csv"          # reserved: external ChatGPT/Claude check
parts = []
for f in sorted(glob.glob(os.path.join(DATA_DIR, "wallstreetbets_*.csv"))):
    if os.path.basename(f) == CHECK_FILE: continue
    d = pd.read_csv(f, usecols=["id", "body", "created_utc", "score"])
    d["ticker"] = os.path.basename(f).split("_")[1]
    parts.append(d)
R = pd.concat(parts, ignore_index=True).drop_duplicates(["ticker", "id"])
R = R[R.body.notna() & ~R.body.isin(["[deleted]", "[removed]"])]
R["n_words"] = R.body.str.split().str.len()
R = R[R.n_words >= MIN_WORDS].copy()
R["day"] = pd.to_datetime(R.created_utc, unit="s", utc=True).dt.tz_convert("America/New_York").dt.strftime("%Y-%m-%d")
print("eligible comments:", len(R), R.ticker.value_counts().to_dict())

rng = np.random.default_rng(SEED)
samp = []
for t, g in R.groupby("ticker"):
    days = g.day.unique(); k = int(np.ceil(N_PER_TICKER / len(days))) + 1
    s = g.sample(frac=1, random_state=SEED).groupby("day").head(k)       # at most k per day, spread over all days
    samp.append(s.sample(min(N_PER_TICKER, len(s)), random_state=SEED))
S = pd.concat(samp, ignore_index=True)

all_days = np.array(sorted(R.day.unique())); perm = rng.permutation(all_days)
n = len(perm); day_split = {d: ("train" if i < 0.70 * n else "val" if i < 0.85 * n else "test") for i, d in enumerate(perm)}
S["split"] = S.day.map(day_split)
S["text"] = S.body.str.slice(0, MAX_CHARS)
print(S.groupby(["ticker", "split"]).size().unstack())
print("days:", pd.Series(day_split).value_counts().to_dict(), "| date range", S.day.min(), "→", S.day.max())

# external check set (no header row in these files)
cd = os.path.join(DATA_DIR, "Comments_w_Labels")
gpt = pd.read_csv(os.path.join(cd, "wallstreetbets_BB_Nov30_2020_Dec9_2020_body_Only_w_ChatGPTLabels.csv"), header=None, names=["body", "gpt"])
cla = pd.read_csv(os.path.join(cd, "wallstreetbets_BB_Nov30_2020_Dec9_2020_body_Only_w_ClaudeLabels.csv"), header=None, names=["body", "claude"])
CHK = gpt.assign(claude=cla.claude.values)
CHK = CHK[pd.to_numeric(CHK.gpt, errors="coerce").notna() & pd.to_numeric(CHK.claude, errors="coerce").notna()].astype({"gpt": float, "claude": float}).reset_index(drop=True)
CHK["text"] = CHK.body.astype(str).str.slice(0, MAX_CHARS); CHK["ticker"] = "BB"
print("check set:", len(CHK))
print(S[["ticker", "day", "text"]].sample(5, random_state=1).to_string(index=False)[:1500])

eligible comments: 856924 {'GME': 424618, 'AMC': 276322, 'BB': 155984}
split   test  train  val
ticker                  
AMC      275   1404  321
BB       264   1439  297
GME      328   1412  260
days: {'train': 129, 'val': 27, 'test': 27} | date range 2021-01-01 → 2021-07-04
check set: 2556
ticker        day                                                                                             text
   AMC 2021-02-06 I’ll let you know! I’ve been thinking of what I’m gonna yolo into before selling my GME and amc.
    BB 2021-05-29                                                                               Yes buying more BB
    BB 2021-04-13         Aapl has been making a fucking killing\n\n\nI should dump my bb and just put it in $aapl
    BB 2021-05-14                              Don't worry bb I'm about to play guitar with my feet just for you 😘
    BB 2021-05-28                                              BB and PLTR are Selling at a discount at Lunch Time


## 2. Load Qwen2.5-7B-Instruct (4-bit), same as 03a

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"
PROMPT_VERSION = "reddit_v1"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.padding_side = "left"
if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, quantization_config=bnb, device_map="auto", low_cpu_mem_usage=True)
model.eval()
print(f"Loaded {MODEL_NAME}. GPU memory used: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/27.8k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Loaded Qwen/Qwen2.5-7B-Instruct. GPU memory used: 5.9 GB


## 3. Reddit prompt and classifier (next-token probabilities)

In [ ]:
COMPANY = {"GME": "GameStop", "AMC": "AMC Entertainment", "BB": "BlackBerry"}
LABELS = ["bearish", "neutral", "bullish"]
SYSTEM = ("You are a careful financial analyst. You classify the sentiment of retail-investor comments from the "
          "Reddit forum r/wallstreetbets for investors in a specific stock. You understand the forum's slang: "
          "rockets, moon, diamond hands, holding, buying the dip, calls and tendies usually signal a bullish view; "
          "puts, shorting, selling, bagholder, rug pull and 'it's over' usually signal a bearish view. "
          "Jokes, memes, questions and off-topic remarks without a clear view on the stock are neutral.")

def build_prompt(text, ticker):
    user = (f"Stock: {ticker} ({COMPANY[ticker]})\n"
            f"Comment: {text}\n\n"
            f"From the point of view of an investor holding {ticker}, does this comment express a bearish, neutral, or bullish view of {ticker}?\n"
            f"- bullish: optimistic about {ticker}, buying/holding, expects the price to rise\n"
            f"- bearish: pessimistic about {ticker}, selling/shorting, expects the price to fall\n"
            f"- neutral: no clear view on {ticker}, off-topic, a question, or mixed\n"
            "Answer with exactly one word: bearish, neutral, or bullish.")
    messages = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": user}]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

def first_token_ids(word):
    ids = set()
    for v in [word, word.capitalize(), " " + word, " " + word.capitalize()]:
        ids.add(tokenizer.encode(v, add_special_tokens=False)[0])
    return sorted(ids)
LABEL_TOKEN_IDS = {lab: first_token_ids(lab) for lab in LABELS}

@torch.no_grad()
def llm_probs(texts, tickers, batch_size=16):
    order = np.argsort([len(t) for t in texts])            # similar lengths together = less padding = faster
    out = np.zeros((len(texts), 3), dtype=np.float32)
    for i in range(0, len(texts), batch_size):
        b = order[i:i + batch_size]
        enc = tokenizer([build_prompt(texts[j], tickers[j]) for j in b], return_tensors="pt", padding=True).to(model.device)
        lp = torch.log_softmax(model(**enc).logits[:, -1, :].float(), dim=-1)
        per = torch.stack([torch.logsumexp(lp[:, ids], dim=-1) for ids in LABEL_TOKEN_IDS.values()], dim=-1)
        out[b] = torch.softmax(per, dim=-1).cpu().numpy()
    return out

def calibrate(P, T=4.0, w_neutral=4.0):                    # same calibration as 03a
    z = np.log(np.clip(P, 1e-12, 1.0)) / T + np.log(np.array([1.0, w_neutral, 1.0]))
    Q = np.exp(z - z.max(axis=1, keepdims=True)); return Q / Q.sum(axis=1, keepdims=True)

demo = S.sample(8, random_state=3)
p = calibrate(llm_probs(demo.text.tolist(), demo.ticker.tolist()))
print(pd.DataFrame({"ticker": demo.ticker.values, "comment": demo.text.str.slice(0, 90).values,
                    "label": np.array(LABELS)[p.argmax(1)], "conf": p.max(1).round(2)}).to_string(index=False))

ticker                                                                                    comment   label  conf
   AMC                                                                         So amc squeeze yet neutral  0.85
   AMC Clover is not a household name wtf well it’s a good thing I made all that money already an bullish  0.77
   AMC     The whole point of throwing these in is to take momentum and money away from AMC N GME bearish  0.71
    BB Even if BB goes to $20 and somehow maintains that as a base, a lot of us apes will be happ bullish  0.56
   AMC Amc straight up recovered my portfolio from embarrassingly massive losses last week. Thank bullish  0.83
    BB                                                     I'm holding BB til it hits $47\n🚀🚀🚀🚀🚀🚀 bullish  0.79
    BB Setting my calendar for May 3rd, 2027 when BB finally announces the patent deal. Stock wil bullish  0.80
   AMC                                   if you didn't make money on amc i don't feel bad for you bearis

## 4. Label everything (checkpointed)

In [ ]:
todo = pd.concat([S[["ticker", "id", "text"]].assign(src="sample"),
                  CHK[["ticker", "text"]].assign(id=[f"chk_{i}" for i in range(len(CHK))], src="check")], ignore_index=True)
todo["key"] = todo.src + "|" + todo.ticker + "|" + todo.id.astype(str)
CKPT = os.path.join(DATA_DIR, f"llm_labels_reddit_partial_{PROMPT_VERSION}.csv")
done = pd.read_csv(CKPT) if os.path.exists(CKPT) else pd.DataFrame(columns=["key", "p_bear_raw", "p_neu_raw", "p_bull_raw"])
rest = todo[~todo.key.isin(done.key)]
print(f"already labeled {len(done)}, to do {len(rest)}")
CHUNK, t0 = 500, time.time()
for s in range(0, len(rest), CHUNK):
    part = rest.iloc[s:s + CHUNK]
    P = llm_probs(part.text.tolist(), part.ticker.tolist())
    done = pd.concat([done, pd.DataFrame({"key": part.key.values, "p_bear_raw": P[:, 0], "p_neu_raw": P[:, 1], "p_bull_raw": P[:, 2]})], ignore_index=True)
    done.to_csv(CKPT, index=False)
    el = time.time() - t0; k = s + len(part)
    print(f"  {k:,}/{len(rest):,} | {k/el:.1f} texts/s | ETA {(len(rest)-k)/(k/el)/60:.0f} min")
print("done:", len(done))

already labeled 0, to do 8556


/tmp/ipykernel_529/427802288.py:12: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  done = pd.concat([done, pd.DataFrame({"key": part.key.values, "p_bear_raw": P[:, 0], "p_neu_raw": P[:, 1], "p_bull_raw": P[:, 2]})], ignore_index=True)


  500/8,556 | 2.6 texts/s | ETA 52 min
  1,000/8,556 | 2.6 texts/s | ETA 49 min
  1,500/8,556 | 2.6 texts/s | ETA 46 min
  2,000/8,556 | 2.6 texts/s | ETA 42 min
  2,500/8,556 | 2.6 texts/s | ETA 39 min
  3,000/8,556 | 2.6 texts/s | ETA 36 min
  3,500/8,556 | 2.6 texts/s | ETA 33 min
  4,000/8,556 | 2.6 texts/s | ETA 30 min
  4,500/8,556 | 2.5 texts/s | ETA 27 min
  5,000/8,556 | 2.5 texts/s | ETA 24 min
  5,500/8,556 | 2.5 texts/s | ETA 20 min
  6,000/8,556 | 2.5 texts/s | ETA 17 min
  6,500/8,556 | 2.5 texts/s | ETA 14 min
  7,000/8,556 | 2.5 texts/s | ETA 10 min
  7,500/8,556 | 2.5 texts/s | ETA 7 min
  8,000/8,556 | 2.5 texts/s | ETA 4 min
  8,500/8,556 | 2.6 texts/s | ETA 0 min
  8,556/8,556 | 2.6 texts/s | ETA 0 min
done: 8556


## 5. Calibrate, inspect, check against ChatGPT / Claude, save

In [ ]:
lab = todo.merge(done, on="key", how="left")
Praw = lab[["p_bear_raw", "p_neu_raw", "p_bull_raw"]].to_numpy()
Pc = calibrate(Praw)
lab["p_bear"], lab["p_neu"], lab["p_bull"] = Pc[:, 0], Pc[:, 1], Pc[:, 2]
lab["llm_label"] = np.array(LABELS)[Pc.argmax(1)]; lab["llm_label_raw"] = np.array(LABELS)[Praw.argmax(1)]
lab["llm_confidence"] = Pc.max(1)

smp = lab[lab.src == "sample"].merge(S[["ticker", "id", "created_utc", "score", "day", "split"]], on=["ticker", "id"], how="left")
print("raw label mix:", smp.llm_label_raw.value_counts(normalize=True).round(3).to_dict())
print("calibrated label mix:", smp.llm_label.value_counts(normalize=True).round(3).to_dict())
print(pd.crosstab(smp.ticker, smp.llm_label, normalize="index").round(2))
print("\nby split:\n", pd.crosstab(smp.split, smp.llm_label))
for l in LABELS:
    ex = smp[smp.llm_label == l].nlargest(4, "llm_confidence")
    print(f"\n--- most confident {l} ---"); print("\n".join(f"[{t}] {x[:140]!r}" for t, x in zip(ex.ticker, ex.text)))

chk = lab[lab.src == "check"].reset_index(drop=True)
CHK["qwen"] = chk.p_bull.values - chk.p_bear.values
rows = []
def sgn(x, thr=0.15): return np.where(x > thr, 1, np.where(x < -thr, -1, 0))
from sklearn.metrics import cohen_kappa_score
for a, b in [("qwen", "gpt"), ("qwen", "claude"), ("gpt", "claude")]:
    rho, p = stats.spearmanr(CHK[a], CHK[b])
    rows.append({"pair": f"{a} vs {b}", "spearman": rho, "3-class agreement": (sgn(CHK[a]) == sgn(CHK[b])).mean(), "kappa": cohen_kappa_score(sgn(CHK[a]), sgn(CHK[b]))})
print("\nExternal check, 2,555 BB comments (Nov–Dec 2020):\n", pd.DataFrame(rows).round(3).to_string(index=False))

out = smp[["ticker", "id", "created_utc", "day", "split", "score", "text", "llm_label", "p_bear", "p_neu", "p_bull", "llm_confidence",
           "llm_label_raw", "p_bear_raw", "p_neu_raw", "p_bull_raw"]].assign(llm_model=MODEL_NAME, prompt_version=PROMPT_VERSION, calib_T=4, calib_w=4)
out.to_csv(os.path.join(DATA_DIR, "llm_labels_reddit.csv"), index=False)
CHK[["text", "gpt", "claude"]].assign(p_bear=chk.p_bear.values, p_neu=chk.p_neu.values, p_bull=chk.p_bull.values, qwen=CHK.qwen).to_csv(
    os.path.join(DATA_DIR, "llm_labels_reddit_bbcheck.csv"), index=False)
print("saved llm_labels_reddit.csv:", out.shape, "and llm_labels_reddit_bbcheck.csv")

raw label mix: {'bullish': 0.392, 'neutral': 0.321, 'bearish': 0.287}
calibrated label mix: {'neutral': 0.525, 'bullish': 0.301, 'bearish': 0.174}
llm_label  bearish  bullish  neutral
ticker                              
AMC           0.20     0.28     0.52
BB            0.18     0.29     0.54
GME           0.15     0.34     0.51

by split:
 llm_label  bearish  bullish  neutral
split                               
test           133      251      483
train          739     1285     2231
val            175      268      435

--- most confident bearish ---
[GME] "I find nothing strange about it. These bags are not empty. They have your money it, and it's on fire. And I am telling you to get your money"
[BB] 'BB is going the other direction, namely to hell'
[GME] 'GME is done already, take the L and move the fuck on already.'
[AMC] 'Amc going to tank hard tomorrow'

--- most confident neutral ---
[BB] 'Can someone tell me what’s going on with $BB? 🤔'
[AMC] 'I think you mispelled AMC'
[AMC